# Train the FD001 AE pilot. Final five-detector results are completed in 05B.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, time
import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
    precision_score, recall_score, f1_score, confusion_matrix)


DATA_DIR = Path("data")
PROTOCOL_DIR = Path("revision_outputs") / "02_FD001_protocol"
OUTPUT_ROOT = Path("revision_outputs") / "04_FD001_representation_pilot"
FOLD = 1
SEED = 42
SENSORS = [f"Sensor{i}" for i in range(1,22)]
COLUMNS = ["UnitNumber", "Time", "OpSet1", "OpSet2", "OpSet3"] + SENSORS
EXPECTED_PROTOCOL = "880c712efeb9fdbdeec4d940c24247f7b1bab5c69a00e098106f88fc9c6774e8"
CANDIDATES = {"IF": [{"n_estimators":n} for n in [100,200,400]],
              "LOF": [{"n_neighbors":n} for n in [20,50,100]],
              "kNN": [{"n_neighbors":n} for n in [20,50,100]]}


In [ ]:
try:
    import tensorflow as tf
except ImportError as exc:
    raise ImportError("Bu notebook TensorFlow gerektiriyor. TensorFlow kurulu kernel'i seçin; yoksa bu hata mesajını paylaşın.") from exc
print("TensorFlow:",tf.__version__)
print("Python:",platform.python_version(),"| scikit-learn:",sklearn.__version__)


In [ ]:
def require(ok, message):
    if not bool(ok): raise ValueError(message)

def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def choose_threshold(y, scores):
    require(np.isfinite(scores).all() and len(np.unique(y))==2, "Geçersiz seçim skorları/etiketleri.")
    p,r,t=precision_recall_curve(y, scores)
    f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(t),where=(p[:-1]+r[:-1])>0)

    i=np.flatnonzero(np.isclose(f,f.max(),rtol=0,atol=1e-12))[-1]
    return float(t[i])

def fit_candidate(method, params, X):
    if method=="IF":
        model=IsolationForest(**params,max_samples="auto",contamination="auto",random_state=SEED,n_jobs=-1)
    elif method=="LOF":
        require(params["n_neighbors"]<len(X), "LOF için yetersiz fit örneği.")
        model=LocalOutlierFactor(**params,novelty=True,contamination="auto",metric="euclidean",n_jobs=-1)
    else:
        require(params["n_neighbors"]<=len(X), "kNN için yetersiz fit örneği.")
        model=NearestNeighbors(**params,metric="euclidean",n_jobs=-1)
    return model.fit(X)

def score(method, model, X):


    values = (model.kneighbors(X,return_distance=True)[0][:,-1]
              if method=="kNN" else -model.score_samples(X))
    require(np.isfinite(values).all(), "Sonlu olmayan model skoru.")
    return values

def metrics(y,scores,threshold):
    pred=(scores>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {"AP":float(average_precision_score(y,scores)),
            "ROC_AUC":float(roc_auc_score(y,scores)),
            "precision":float(precision_score(y,pred,zero_division=0)),
            "recall":float(recall_score(y,pred,zero_division=0)),
            "F1":float(f1_score(y,pred,zero_division=0)),
            "Macro_F1":float(f1_score(y,pred,average="macro",zero_division=0)),
            "TP":int(tp),"FP":int(fp),"TN":int(tn),"FN":int(fn)}


In [ ]:
summary_path=PROTOCOL_DIR/"protocol_summary.json"
require(summary_path.is_file(), f"Dosya yok: {summary_path}")
stage02=json.loads(summary_path.read_text(encoding="utf-8"))
require(stage02["status"]=="PASS" and stage02["canonical_protocol_sha256"]==EXPECTED_PROTOCOL,
        "Kontrol edilen 02 protokolüyle uyuşmuyor.")
for name, expected in stage02["output_file_sha256"].items():
    require((PROTOCOL_DIR/name).is_file() and digest(PROTOCOL_DIR/name)==expected,
            f"02 çıktısı eksik/değişmiş: {name}")
protocol=json.loads((PROTOCOL_DIR/"experiment_protocol.json").read_text(encoding="utf-8"))
canonical=json.dumps(protocol,sort_keys=True,ensure_ascii=False,separators=(",",":"))
require(hashlib.sha256(canonical.encode()).hexdigest()==EXPECTED_PROTOCOL,"Protokol içeriği farklı.")
train_path=DATA_DIR/"train_FD001.txt"
require(train_path.is_file(),f"Dosya yok: {train_path}")
require(digest(train_path)==protocol["source_raw_file_sha256_from_stage01"]["train_FD001.txt"],
        "Ham eğitim dosyası 01 aşamasındaki dosyadan farklı.")
raw=pd.read_csv(train_path,sep=r"\s+",header=None)
require(raw.shape==(20631,26),"Ham veri boyutu farklı.")
raw.columns=COLUMNS
require(np.isfinite(raw.to_numpy(dtype=float)).all(),"Ham veride eksik/sonsuz değer.")
reference=pd.read_csv(PROTOCOL_DIR/"train_evaluation_reference.csv")
require(len(reference)==len(raw) and np.array_equal(reference.source_row,np.arange(len(raw))),"Metadata sırası farklı.")
require(np.array_equal(raw[["UnitNumber","Time"]].to_numpy(),reference[["UnitNumber","Time"]].to_numpy()),
        "Sensör ve metadata satırları eşleşmiyor.")
require(np.array_equal(reference.late_life_le_30,reference.RUL.le(30).astype(int)),"Referans etiketi farklı.")
splits=pd.read_csv(PROTOCOL_DIR/"engine_splits.csv")
fold=splits.loc[splits.scheme.eq("cv") & splits.fold.eq(FOLD)].copy()
require(fold.engine_key.is_unique and len(fold)==100,"Motor rol çakışması/eksik motor.")
require(fold.role.value_counts().to_dict()=={"fit":70,"early_stop":10,"selection":10,"outer_valid":10},
        "Beklenen motor sayıları farklı.")
role_map=fold.set_index("engine_key").role
roles=reference.engine_key.map(role_map)
require(roles.notna().all(),"Atanmamış motor var.")
masks={role:roles.eq(role).to_numpy() for role in ["fit","early_stop","selection","outer_valid"]}
X=raw[SENSORS].to_numpy(dtype=np.float32)
scaler=StandardScaler().fit(X[masks["fit"]])
Xs={role:scaler.transform(X[mask]) for role,mask in masks.items()}
y_selection=reference.loc[masks["selection"],"late_life_le_30"].to_numpy(dtype=int)
require(len(np.unique(y_selection))==2,"Seçim kümesi tek sınıf.")


run_id=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUT=OUTPUT_ROOT/run_id
OUT.mkdir(parents=True,exist_ok=False)
plan={"stage":"04_representation_pilot","fold":FOLD,"seed":SEED,"candidates":CANDIDATES,
      "representation":"StandardScaler on fit engines; all 21 sensors",
      "selection":"AP; ties keep first candidate in listed order",
      "threshold":"positive F1 on selection engines; ties choose higher threshold; score >= threshold",
      "score_definitions":{"IF":"negative score_samples","LOF":"negative novelty score_samples",
                           "kNN":"kth Euclidean distance to fit samples"},
      "protocol_sha256":EXPECTED_PROTOCOL,"raw_train_sha256":digest(train_path)}
plan["representations"]=["Raw", "AE_latent21", "AE_reconstruction_MSE"]
plan["AE"]={"widths":[21,42,21,21,21,42,21],"hidden_activation":"relu",
            "latent_activation":"linear","output_activation":"linear",
            "optimizer":"Adam","learning_rate":0.001,"loss":"MSE",
            "batch_size":256,"max_epochs":100,"early_stopping_patience":10,
            "early_stopping_role":"early_stop","latent_scaler_fit_role":"fit"}
(OUT/"pilot_plan.json").write_text(json.dumps(plan,indent=2),encoding="utf-8")
print("Girdi kontrolü tamamlandı. Çıktı klasörü:",OUT.resolve())


In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
inputs=tf.keras.Input(shape=(21,),name="sensor_input")
h=tf.keras.layers.Dense(42,activation="relu",name="encoder_42")(inputs)
h=tf.keras.layers.Dense(21,activation="relu",name="encoder_21")(h)
latent=tf.keras.layers.Dense(21,activation="linear",name="latent")(h)
h=tf.keras.layers.Dense(21,activation="relu",name="decoder_21")(latent)
h=tf.keras.layers.Dense(42,activation="relu",name="decoder_42")(h)
outputs=tf.keras.layers.Dense(21,activation="linear",name="reconstruction")(h)
ae=tf.keras.Model(inputs,outputs,name="AE21")
encoder=tf.keras.Model(inputs,latent,name="encoder21")
ae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),loss="mse")
early=tf.keras.callbacks.EarlyStopping(monitor="val_loss",mode="min",patience=10,restore_best_weights=True)
started=time.perf_counter()
history=ae.fit(Xs["fit"],Xs["fit"],validation_data=(Xs["early_stop"],Xs["early_stop"]),
               epochs=100,batch_size=256,shuffle=True,callbacks=[early],verbose=2)
training_seconds=time.perf_counter()-started
require(np.isfinite(history.history["loss"]).all() and np.isfinite(history.history["val_loss"]).all(),
        "AE eğitiminde sonlu olmayan loss oluştu.")
ae.save(OUT/"autoencoder.keras")
encoder.save(OUT/"encoder.keras")
pd.DataFrame(history.history).assign(epoch=np.arange(1,len(history.history["loss"])+1)).to_csv(OUT/"ae_history.csv",index=False)
Z={role:encoder.predict(Xs[role],batch_size=256,verbose=0) for role in ["fit","selection","outer_valid"]}
for role,values in Z.items():
    require(values.shape==(len(Xs[role]),21) and np.isfinite(values).all(),f"Geçersiz latent çıktı: {role}")
latent_scaler=StandardScaler().fit(Z["fit"])
Zs={role:latent_scaler.transform(values) for role,values in Z.items()}
representations={"Raw":Xs,"AE_latent21":Zs}
ae_scores={}
for role in ["selection","outer_valid"]:
    reconstructed=ae.predict(Xs[role],batch_size=256,verbose=0)
    ae_scores[role]=np.mean(np.square(Xs[role]-reconstructed),axis=1)
    require(np.isfinite(ae_scores[role]).all(),"Geçersiz AE reconstruction skoru.")
ae_info={"epochs_run":len(history.history["loss"]),
         "best_epoch":int(np.argmin(history.history["val_loss"])+1),
         "best_early_stop_loss":float(min(history.history["val_loss"])),
         "parameter_count":int(ae.count_params()),"training_seconds":training_seconds,
         "latent_fit_std":Z["fit"].std(axis=0).tolist(),
         "selection_reconstruction_score_quantiles":np.quantile(ae_scores["selection"],[0,.5,.9,.99,1]).tolist()}
print("AE eğitimi tamamlandı. Çalışan epoch:",ae_info["epochs_run"],"| En iyi epoch:",ae_info["best_epoch"])


In [ ]:

chosen={}
trials=[]
for representation,features in representations.items():
    for method,candidates in CANDIDATES.items():
        best=None
        for i,params in enumerate(candidates):
            started=time.perf_counter()
            model=fit_candidate(method,params,features["fit"])
            selection_scores=score(method,model,features["selection"])
            ap=float(average_precision_score(y_selection,selection_scores))
            trials.append({"representation":representation,"method":method,"candidate_order":i,
                           "params":params,"selection_AP":ap,"seconds":time.perf_counter()-started})
            if best is None or ap>best["selection_AP"]:
                best={"model":model,"params":params,"selection_AP":ap,"scores":selection_scores}
            print(representation,method,params,"selection AP =",round(ap,6),flush=True)
        best["threshold"]=choose_threshold(y_selection,best["scores"])
        chosen[(representation,method)]=best
chosen[("AE_reconstruction","MSE")]={"params":{},
    "selection_AP":float(average_precision_score(y_selection,ae_scores["selection"])),
    "threshold":choose_threshold(y_selection,ae_scores["selection"])}
(OUT/"selection_trials.json").write_text(json.dumps(trials,indent=2),encoding="utf-8")
print("Seçimler tamamlandı. Dış fold raporlanıyor.")


In [ ]:
held=reference.loc[masks["outer_valid"]].reset_index(drop=True)
results=[]
frames=[]
for (representation,method),best in chosen.items():
    scores=(ae_scores["outer_valid"] if method=="MSE" else
            score(method,best["model"],representations[representation]["outer_valid"]))
    for window in [20,30,40]:
        y=held[f"late_life_le_{window}"].to_numpy(dtype=int)
        require(len(np.unique(y))==2,"Dış referans tek sınıf.")
        results.append({"representation":representation,"method":method,"RUL_window":window,
                        "selected_params":best["params"],"selection_AP_primary":best["selection_AP"],
                        "threshold":best["threshold"],**metrics(y,scores,best["threshold"])})
    frame=held[["sample_key","engine_key","UnitNumber","Time","RUL",
                "late_life_le_20","late_life_le_30","late_life_le_40"]].copy()
    frame["representation"]=representation
    frame["method"]=method
    frame["score"]=scores
    frame["threshold"]=best["threshold"]
    frame["prediction"]=(scores>=best["threshold"]).astype(int)
    frames.append(frame)
pd.concat(frames,ignore_index=True).to_csv(OUT/"outer_predictions.csv",index=False)
pd.DataFrame(results).to_csv(OUT/"representation_metrics.csv",index=False)
scaler_info={"input":{"features":SENSORS,"mean":scaler.mean_.tolist(),"scale":scaler.scale_.tolist()},
             "latent":{"mean":latent_scaler.mean_.tolist(),"scale":latent_scaler.scale_.tolist()},
             "fit_rows":int(masks["fit"].sum())}
(OUT/"scaler_parameters.json").write_text(json.dumps(scaler_info,indent=2),encoding="utf-8")
primary=[r for r in results if r["RUL_window"]==30]
comparisons=[]
for method in CANDIDATES:
    raw_result=next(r for r in primary if r["representation"]=="Raw" and r["method"]==method)
    latent_result=next(r for r in primary if r["representation"]=="AE_latent21" and r["method"]==method)
    comparisons.append({"method":method,"raw_AP":raw_result["AP"],"latent_AP":latent_result["AP"],
                        "delta_AP":latent_result["AP"]-raw_result["AP"],
                        "raw_F1":raw_result["F1"],"latent_F1":latent_result["F1"],
                        "delta_F1":latent_result["F1"]-raw_result["F1"]})


In [ ]:
PREVIOUS_RAW_RESULTS = [{'method': 'IF', 'RUL_window': 30, 'selected_params': {'n_estimators': 400}, 'selection_AP_primary': 0.7957486368454624, 'threshold': 0.47326042503387084, 'AP': 0.7708424039543613, 'ROC_AUC': 0.9415756891655936, 'precision': 0.660377358490566, 'recall': 0.7903225806451613, 'F1': 0.7195301027900147, 'Macro_F1': 0.8332151486983157, 'TP': 245, 'FP': 126, 'TN': 1703, 'FN': 65}, {'method': 'LOF', 'RUL_window': 30, 'selected_params': {'n_neighbors': 100}, 'selection_AP_primary': 0.1608048672224708, 'threshold': 0.9772202372550964, 'AP': 0.16222260976337305, 'ROC_AUC': 0.49262509038960123, 'precision': 0.1482194417709336, 'recall': 0.9935483870967742, 'F1': 0.25795644891122277, 'Macro_F1': 0.1601951556725426, 'TP': 308, 'FP': 1770, 'TN': 59, 'FN': 2}, {'method': 'kNN', 'RUL_window': 30, 'selected_params': {'n_neighbors': 100}, 'selection_AP_primary': 0.5133043194351089, 'threshold': 2.097911834716797, 'AP': 0.6109647146257249, 'ROC_AUC': 0.9021975696220393, 'precision': 0.4634615384615385, 'recall': 0.7774193548387097, 'F1': 0.5807228915662651, 'Macro_F1': 0.7398974086601627, 'TP': 241, 'FP': 279, 'TN': 1550, 'FN': 69}]

reproduction=[]
for previous in PREVIOUS_RAW_RESULTS:
    current=next(r for r in primary if r["representation"]=="Raw" and r["method"]==previous["method"])
    reproduction.append({"method":previous["method"],"AP_difference_from_03":current["AP"]-previous["AP"],
                         "F1_difference_from_03":current["F1"]-previous["F1"],
                         "same_selected_params":current["selected_params"]==previous["selected_params"]})
summary={"stage":"04_representation_pilot","status":"PASS","fold":FOLD,"seed":SEED,
         "scope":"single-fold pilot; RUL late-life proxy, not annotated anomaly onset",
         "protocol_sha256":EXPECTED_PROTOCOL,
         "versions":{"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__,
                     "sklearn":sklearn.__version__,"tensorflow":tf.__version__},
         "roles":{role:{"engines":int(reference.loc[mask,"engine_key"].nunique()),"rows":int(mask.sum())}
                  for role,mask in masks.items()},
         "test_files_read":False,"AE_trained":True,"AE":ae_info,
         "outer_primary_prevalence":float(held.late_life_le_30.mean()),
         "primary_results":primary,"raw_vs_latent":comparisons,"raw_reproduction_check":reproduction,
         "sensitivity_results":[r for r in results if r["RUL_window"]!=30],
         "selection_trials":trials,
         "output_sha256":{p.name:digest(p) for p in OUT.iterdir() if p.is_file()}}
(OUT/"representation_summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
print(pd.DataFrame(primary)[["representation","method","AP","ROC_AUC","precision","recall","F1"]].to_string(index=False))
print("REPRESENTATION PILOT: PASS — çalışma tamamlandı; bu ifade AE'nin üstün olduğu anlamına gelmez.")
print("Gönderilecek dosya:",(OUT/"representation_summary.json").resolve())
